## 第7章 再谈抽象(类)

### 1.面向对象编程

- **类**：是创建对象的蓝图或模板，包含了该类对象所共有的属性(数据)和方法(行为)。
- **对象**：根据类创建的具体实例，每个对象都有自己独立的属性值，并可以使用类中定义的方法。
    - 关系：类是抽象的概念，对象是具体的实体。类本身不占用内存执行具体操作，而对象在内存中真实存在并承载数据。
- **多态**：多态意味着“有多种形态”，不同对象对同一操作有不同响应。不关心对象是什么类型，只关心对象能不能做某件事情。
- **封装**：封装就是对外隐藏对象内部细节，只通过方法交互，不直接碰数据。多态让你无需知道对象的**类型**，封装让你无需知道对象的**结构**。
- **继承**：继承就是基于已有类创建新类，自动获得超类的方法，也可以重写或添加新的方法。

### 2.创建类

- **类的定义**：使用`class`关键字定义一个类，后面跟着类名(通常首字母大写)和冒号`:`。
    - `self`：指向当前对象本身，用于访问对象的属性和方法。
    - 属性：类的实例变量，用于存储对象的状态。
        - Python属性，所有属性属性，所有属性都是公有的。
            - 可以让属性名称以两个下划线`__`开头变为私有属性，但也可以通过`_类名__属性名`来访问。
            - 还可以通过命名约定来说明为“私有属性”，即在属性名前添加下划线`_`。从`from module import *`中不会被导入。
    - 方法：类的实例函数，用于定义对象的行为。
        - 在Python中也没有私有方法。也是通过在方法名前添加下划线`_`这种约定来声明为“私有方法”。


In [ ]:
# 定义类
class Person:

    # 方法
    def set_name(self, name):
        self.name = name  # 设置属性

    def get_name(self):
        return self.name

    def greet(self):
        print(f"Hello, I am {self.name}!")

foo = Person()  # 实例化类
foo.set_name("Anakin Skywalker")  # 设置属性
foo.greet()  # 调用方法

- **类定义的本质**：`class`语句会创建一个独立的命名空间，类定义中的所有代码都会在这个命名空间内执行。
    - 类级变量：在类内定义的变量属于类本身，所有实例都可访问。可以通过`类名.变量名`或`实例.变量名`访问。
    - 实例属性遮盖类级变量：如果对某个实例的类级变量赋值，会创建一个新的实例属性，遮盖掉类级变量。


In [ ]:
class MemberCounter:
    members = 0     # 类级变量，所有实例共享
    def init(self):
        MemberCounter.members += 1

m1 = MemberCounter()
m1.init()
print(MemberCounter.members)        # 1
m2 = MemberCounter()
m2.init()
print(MemberCounter.members)        # 2，所有实例共享

print(m1.members)                   # 2
m1.members = 100                    # 在实例m1上创建一个属性members，遮盖掉类级变量members
print(m1.members)                   # 100
print(m2.members)                   # 2

### 3.继承类

- **定义继承**：`class 子类名(父类名1, 父类名2, ...)`。
    - 检查继承关系：`issubclass(子类名, 父类名)`。
        - 查看基类(返回元组)：`类名.__bases__`。
    - 检查实例关系(包括间接实例)：`isinstance(实例, 类名)`。
        - 查看实例所属类：`实例.__class__` 或 `type(实例)`。
    - 多重继承：一个类可以继承自多个父类。
        - 如果多个父类有同名方法，**排在前面的类优先**。⚠️慎用多重继承，它可能带来意外的复杂性。

In [ ]:
# 父类
class Filter:
    def init(self):
        self.blocked = []
    def filter(self, seq):
        return [x for x in seq if x not in self.blocked]

# 子类：在括号中指定父类名
class SpamFilter(Filter):
    def init(self):
        self.blocked = ['SPAM']

s = SpamFilter()
s.init()
print(s.filter(['SPAM', 'SPAM', 'SPAM', 'SPAM', 'eggs', 'bacon', 'SPAM']))      # ['eggs', 'bacon']

print(issubclass(SpamFilter, Filter))      # True，检查SpamFilter是否是Filter的子类
print(SpamFilter.__bases__)                # (<class '__main__.Filter'>,)，获取SpamFilter的基类
print(Filter.__bases__)                    # (<class 'object'>,)，获取Filter的基类

print(isinstance(s, SpamFilter))           # True，检查s是否是SpamFilter的实例
print(isinstance(s, Filter))               # True，检查s是否是Filter的实例
print(s.__class__)                         # <class '__main__.SpamFilter'>，获取s的类
print(type(s))                             # <class '__main__.SpamFilter'>，获取s的类型

### 4.接口与自省

- **接口**：对象对外暴露的方法和属性，Python不强制要求实现特定接口。
- **自省**：对象可以检查自己的状态和行为。
    - `hasattr(obj, name)`：检查对象是否有指定属性。
    - `getattr(obj, name[, default])`：获取对象的属性值。如果属性不存在，返回默认值(default)。
    - `setattr(obj, name, value)`：设置对象的属性值。
    - `obj.__dict__`：查看对象所有属性。
    - `callable(obj)`：检查对象是否可调用(如函数、方法)。

### 5.抽象类

- **抽象类**：不能实例化的类，其职责是定义子类必须实现的一组抽象方法。官方使用 `abc` 模块定义抽象类。
    - **注册机制**：可以使用`register()`方法将一个非子类的类注册为抽象类的子类之一。但是不能保障注册后的类会实现抽象方法。

In [ ]:
from abc import ABC, abstractmethod

class Talker(ABC):   # 抽象类(继承自ABC)，不能实例化
    @abstractmethod  # 装饰器
    def talk(self):  # 抽象方法，必须在子类中实现
        pass

class Knigget(Talker):  # 具体类，实现了talk方法
    def talk(self):
        print("Ni!")

class Herring:
    def talk(self):
        print("Blub!")

class Clam:
    pass

# Talker()      # 报错，抽象类不能实例化
# t = Talker()
# t.talk()      # 报错，抽象类的方法必须在子类中实现

k = Knigget()
h = Herring()
k.talk()        # Ni!
h.talk()        # Blub!

print(isinstance(k, Talker))        # True
print(isinstance(h, Talker))        # False

Talker.register(Herring)            # 注册Herring类为Talker的子类
print(isinstance(h, Talker))        # True

Talker.register(Clam)               # 注册Clam类为Talker的子类
print(issubclass(Clam, Talker))     # True
c = Clam()
print(isinstance(c, Talker))        # True
# c.talk()                          # 报错，Clam类没有实现talk方法，通过注册的子类不能保障实现了抽象方法

### 6. 本章核心知识脉络

```text
面向对象编程
│
├── 三大特性
│   ├── 多态 ⭐ —— 不同对象，同一操作，不同行为
│   │   ├── 鸭子类型：不查类型，只看行为
│   │   └── 慎用：type/isinstance检查（除ABC外）
│   │
│   ├── 封装 —— 隐藏内部细节
│   │   ├── 对象属性（非全局变量）
│   │   ├── __name 私有属性（名称改写）
│   │   └── _name 约定私有
│   │
│   └── 继承 —— 基于已有类创建新类
│       ├── 单继承：class Child(Parent)
│       ├── 多重继承：class C(A, B)  慎用！
│       ├── 方法重写（override）
│       └── MRO（方法解析顺序）
│
├── 类的定义
│   ├── class 语句
│   ├── self 参数 ⭐
│   ├── 类命名空间 → 类级变量
│   └── 实例属性 vs 类属性
│
├── 内省与检查
│   ├── issubclass / isinstance
│   ├── hasattr / getattr / setattr
│   ├── __bases__ / __class__ / __dict__
│   └── type()
│
├── 抽象基类（abc）
│   ├── ABC + @abstractmethod
│   ├── 不能实例化
│   ├── 子类必须实现抽象方法
│   └── register() 注册
│
└── 设计原则
    ├── 相关的放一起
    ├── 对象间保持距离
    ├── 慎用继承
    └── 保持简单
```